# 02 — Regresión lineal: el primer modelo

## Motivación

¿Cuánto vale una casa? Depende del ingreso de la zona, la antigüedad, el número de
cuartos, la ubicación... Podríamos intentar escribir la fórmula a mano, pero no la
conocemos. Lo que sí tenemos son **miles de ejemplos** de casas con sus
características y su precio.

Ese es el cambio de perspectiva del *machine learning*: en lugar de programar la regla,
**aprendemos la función a partir de los datos**.

Formalmente, en el **aprendizaje supervisado** tenemos un dataset de pares
$\{(\mathbf{x}_i, y_i)\}_{i=1}^{n}$ — características y respuesta — y buscamos una
función $f$ tal que $f(\mathbf{x}) \approx y$ para datos **nuevos**. Si $y$ es un número
continuo (un precio, una temperatura) hablamos de **regresión**; si es una categoría
(spam / no spam) hablamos de **clasificación** — eso viene en la sesión 04.

Hoy construimos el modelo más simple y más importante de todos: la **regresión lineal**,
punto de partida de casi todo lo que sigue en el curso, incluidas las redes neuronales.

## Teoría: el modelo lineal

Proponemos que la respuesta es una combinación lineal de las $d$ características:

$$f(\mathbf{x}) = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = \mathbf{w}^\top \mathbf{x} + b$$

Los **parámetros** del modelo son los pesos $\mathbf{w} \in \mathbb{R}^d$ y el sesgo
$b \in \mathbb{R}$. "Aprender" significa encontrar los valores de estos parámetros
que mejor explican los datos.

¿Y qué es "mejor"? Necesitamos una **función de pérdida** que mida qué tan mal predice
el modelo. La elección clásica es el **error cuadrático medio** (MSE):

$$L(\mathbf{w}, b) = \frac{1}{n} \sum_{i=1}^{n} \left( y_i - f(\mathbf{x}_i) \right)^2$$

Aprender = **minimizar la pérdida**. Esta idea — proponer una familia de funciones,
definir una pérdida, y optimizar — es *el* esquema de todo el curso. Solo cambiarán
la familia de funciones y el optimizador.

### Mínimos cuadrados: la solución exacta

La regresión lineal tiene una propiedad que casi ningún otro modelo comparte: el mínimo
se puede calcular con una fórmula cerrada.

Absorbemos el sesgo agregando una columna de unos a los datos: $X \in \mathbb{R}^{n \times (d+1)}$
con $\mathbf{w}$ incluyendo ahora a $b$. La pérdida en forma matricial es

$$L(\mathbf{w}) = \frac{1}{n} \lVert X\mathbf{w} - \mathbf{y} \rVert^2$$

Es una función cuadrática (un paraboloide en el espacio de parámetros): su mínimo está
donde el gradiente se anula. Tomando el gradiente respecto a $\mathbf{w}$ e igualando a cero:

$$\nabla_{\mathbf{w}} L = \frac{2}{n} X^\top (X\mathbf{w} - \mathbf{y}) = 0
\quad\Longrightarrow\quad
\boxed{\; X^\top X \, \mathbf{w}^* = X^\top \mathbf{y} \;}$$

Estas son las **ecuaciones normales**: un sistema lineal de $(d+1)$ ecuaciones cuya
solución $\mathbf{w}^*$ es el mejor ajuste posible. La derivación paso a paso está en el
**Apéndice A**; ahí también explicamos por qué en la práctica se resuelve el sistema
en lugar de invertir $X^\top X$.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Convención del curso: los datasets se descargan a datos/ (fuera del repo)
DATA_DIR = Path("../../datos")

rng = np.random.default_rng(seed=42)

## Un experimento controlado: datos sintéticos

Antes de usar datos reales, hagamos lo que se hace en física: un experimento donde
**conocemos la respuesta correcta**. Generamos datos con una ley conocida más ruido,

$$y = 2.5\,x + 1.0 + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

y verificamos si el método recupera la pendiente $2.5$ y la ordenada $1.0$.
Si un método no funciona en el caso donde sabemos la verdad, no hay razón para
confiar en él con datos reales.

In [ ]:
true_slope, true_intercept, noise_std = 2.5, 1.0, 1.5

n_samples = 80
x = rng.uniform(0, 10, size=n_samples)
y = true_slope * x + true_intercept + rng.normal(0, noise_std, size=n_samples)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=7, opacity=0.7)))
fig.update_layout(
    title="Datos sintéticos: ley lineal + ruido gaussiano",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
)
fig.show()

### Mínimos cuadrados a mano

Resolvemos las ecuaciones normales $X^\top X \mathbf{w} = X^\top \mathbf{y}$ con numpy.
La matriz de diseño $X$ tiene una columna de unos (para el sesgo $b$) y una columna con $x$.

In [ ]:
X = np.column_stack([np.ones(n_samples), x])  # columna de unos absorbe el sesgo

w = np.linalg.solve(X.T @ X, X.T @ y)
b_hat, m_hat = w

print(f"pendiente:  estimada = {m_hat:.3f}   verdadera = {true_slope}")
print(f"ordenada:   estimada = {b_hat:.3f}   verdadera = {true_intercept}")

In [ ]:
x_line = np.linspace(0, 10, 100)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x, y=y, mode="markers", name="datos",
    marker=dict(size=7, opacity=0.6),
))
fig.add_trace(go.Scatter(
    x=x_line, y=m_hat * x_line + b_hat, mode="lines", name="ajuste",
    line=dict(width=2, color="#EF553B"),
))
fig.update_layout(
    title="El ajuste recupera la ley que generó los datos",
    xaxis_title="x",
    yaxis_title="y",
    template="plotly_white",
)
fig.show()

## El mismo modelo con scikit-learn

En la práctica no resolvemos las ecuaciones normales a mano: usamos **scikit-learn**,
que lo hace con métodos numéricamente más estables y nos da un API uniforme:

- `model.fit(X, y)` — aprende los parámetros
- `model.predict(X)` — predice sobre datos nuevos

Ese par `fit`/`predict` es idéntico para *todos* los modelos de la librería —
regresión lineal, árboles, ensambles. Este patrón se repite en todo el curso.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(x.reshape(-1, 1), y)  # sklearn espera X con forma (n_samples, n_features)

print(f"pendiente:  sklearn = {model.coef_[0]:.3f}   a mano = {m_hat:.3f}")
print(f"ordenada:   sklearn = {model.intercept_:.3f}   a mano = {b_hat:.3f}")

## Datos reales: California Housing

Retomamos el problema planteado en la motivación. El dataset **California Housing**
contiene 20,640 distritos censales de California (1990) con 8 características —
ingreso mediano de la zona, antigüedad, cuartos promedio, población, latitud/longitud... —
y el objetivo: el **valor mediano de la vivienda** del distrito, en unidades de \\$100,000.

La primera vez, scikit-learn lo descarga automáticamente a nuestra carpeta `datos/`
(que git ignora — los datos nunca van al repo).

In [ ]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(data_home=DATA_DIR, as_frame=True)
df = housing.frame

print(df.shape)
df.head()

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "A mayor ingreso, mayor precio (con mucha dispersión)",
    "Distribución del objetivo",
))

fig.add_trace(
    go.Scatter(
        x=df["MedInc"], y=df["MedHouseVal"], mode="markers",
        marker=dict(size=3, opacity=0.15), showlegend=False,
    ),
    row=1, col=1,
)
fig.add_trace(
    go.Histogram(x=df["MedHouseVal"], nbinsx=50, showlegend=False),
    row=1, col=2,
)

fig.update_xaxes(title_text="ingreso mediano de la zona [decenas de miles de $]", row=1, col=1)
fig.update_yaxes(title_text="valor mediano de la vivienda [$100k]", row=1, col=1)
fig.update_xaxes(title_text="valor mediano de la vivienda [$100k]", row=1, col=2)
fig.update_yaxes(title_text="distritos", row=1, col=2)

fig.update_layout(template="plotly_white", height=450)
fig.show()

Dos observaciones del panel anterior que conviene registrar **antes** de modelar:

1. La relación ingreso–precio existe pero es ruidosa: ninguna característica sola
   determina el precio.
2. El histograma tiene un pico artificial en \\$500k: el dataset original **recortó**
   los valores altos a ese tope. Los datos reales presentan irregularidades de este
   tipo con frecuencia — mirarlos antes de modelar no es opcional.

## Generalización: la partición train/test

Aquí viene **el concepto más importante del curso**.

Si evaluamos el modelo sobre los mismos datos con los que lo ajustamos, nos engañamos:
es como calificar un examen cuyas respuestas el estudiante ya vio. Lo que nos importa
es la **generalización**: el desempeño sobre datos *nuevos*.

La solución es simple: **apartamos una fracción de los datos** (el conjunto de *test*)
que el modelo no ve durante el entrenamiento, y la usamos solo para evaluar:

- **train** (80%): el modelo ajusta sus parámetros aquí
- **test** (20%): medimos aquí el desempeño que reportamos

Toda evaluación que veas en este curso — y en cualquier publicación científica rigurosa —
sigue esta regla.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

print(f"train: {X_train.shape[0]} distritos   test: {X_test.shape[0]} distritos")

### Métricas de regresión

- **MSE**: $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ — la pérdida que minimizamos, en unidades al cuadrado.
- **RMSE**: $\sqrt{\text{MSE}}$ — en las **unidades del objetivo** (aquí, \\$100k), directamente interpretable.
- **$R^2$**: fracción de la varianza del objetivo que el modelo explica; $1$ es perfecto,
  $0$ es no mejor que predecir el promedio.

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

In [ ]:
from sklearn.metrics import r2_score, root_mean_squared_error

for name, X_split, y_split in [("train", X_train, y_train), ("test ", X_test, y_test)]:
    y_pred = model.predict(X_split)
    rmse = root_mean_squared_error(y_split, y_pred)
    r2 = r2_score(y_split, y_pred)
    print(f"{name}  RMSE = {rmse:.3f} [$100k]   R² = {r2:.3f}")

Train y test dan casi lo mismo — el modelo lineal es tan simple que no logra memorizar
nada. Este resultado es relevante: en la próxima sesión veremos qué pasa cuando el
modelo es lo bastante flexible para que train y test **se separen** — el *sobreajuste*.

In [ ]:
y_pred_test = model.predict(X_test)
lims = [0, 5.2]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test, y=y_pred_test, mode="markers", name="distritos (test)",
    marker=dict(size=4, opacity=0.2),
))
fig.add_trace(go.Scatter(
    x=lims, y=lims, mode="lines", name="predicción perfecta",
    line=dict(width=2, color="#EF553B"),
))
fig.update_layout(
    title="Predicho vs. real (test)",
    xaxis_title="valor real [$100k]",
    yaxis_title="valor predicho [$100k]",
    xaxis=dict(range=lims), yaxis=dict(range=lims, scaleanchor="x"),
    template="plotly_white",
    width=550, height=550,
)
fig.show()

### ¿Qué aprendió el modelo?

A diferencia de los modelos que veremos después, la regresión lineal es totalmente
transparente: sus parámetros *son* la explicación. Cada coeficiente dice cuánto cambia
la predicción por unidad de esa característica, con las demás fijas.

**Advertencia**: los coeficientes están en las unidades de cada característica
(personas, grados de latitud, decenas de miles de dólares...), así que **sus magnitudes
no son comparables entre sí** todavía. Para compararlas hay que llevar las
características a una escala común — lo haremos en la sesión 03 con `StandardScaler`.

In [ ]:
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

fig = go.Figure()
fig.add_trace(go.Bar(x=coefs.values, y=coefs.index, orientation="h"))
fig.add_vline(x=0, line_width=1, line_color="black")
fig.update_layout(
    title="Coeficientes del modelo (sin escalar — no comparar magnitudes)",
    xaxis_title="coeficiente [en unidades de cada variable]",
    template="plotly_white",
)
fig.show()

## Ejercicio

Trabaja en una copia de este notebook dentro de `mi-trabajo/`.

1. **Una sola característica.** Entrena una regresión usando únicamente `MedInc`
   y evalúa en test. ¿Cuánto $R^2$ pierdes respecto al modelo con las 8 características?
   ¿Qué te dice eso sobre cuánta información aporta el ingreso por sí solo?

2. **Cuando el modelo se queda corto.** Genera datos sintéticos con
   $y = x^2 + \varepsilon$ para $x \in [-3, 3]$ y ajusta una regresión lineal.
   Grafica los datos y la recta ajustada. Describe *qué* falla y *por qué* —
   la respuesta formal es el tema de la próxima sesión.

3. **Reto.** Implementa el $R^2$ a mano con numpy (sin `sklearn.metrics`) y verifica
   que coincide con `r2_score` en el conjunto de test.

## Apéndice A — Derivación de las ecuaciones normales

Expandimos la pérdida (ignorando el factor $\tfrac{1}{n}$, que no mueve el mínimo):

$$L(\mathbf{w}) = \lVert X\mathbf{w} - \mathbf{y} \rVert^2
= (X\mathbf{w} - \mathbf{y})^\top (X\mathbf{w} - \mathbf{y})
= \mathbf{w}^\top X^\top X \mathbf{w} - 2\, \mathbf{y}^\top X \mathbf{w} + \mathbf{y}^\top \mathbf{y}$$

Usamos dos identidades del gradiente matricial (con $A$ simétrica):

$$\nabla_{\mathbf{w}} \left( \mathbf{w}^\top A \mathbf{w} \right) = 2 A \mathbf{w},
\qquad
\nabla_{\mathbf{w}} \left( \mathbf{c}^\top \mathbf{w} \right) = \mathbf{c}$$

Con $A = X^\top X$ (simétrica por construcción) y $\mathbf{c} = 2 X^\top \mathbf{y}$:

$$\nabla_{\mathbf{w}} L = 2 X^\top X \mathbf{w} - 2 X^\top \mathbf{y} = 0
\quad\Longrightarrow\quad
X^\top X \mathbf{w}^* = X^\top \mathbf{y}$$

Como $L$ es convexa (su hessiana $2X^\top X$ es semidefinida positiva), este punto
crítico es el mínimo global.

**Nota numérica.** En los textos aparece $\mathbf{w}^* = (X^\top X)^{-1} X^\top \mathbf{y}$,
pero **invertir la matriz es mala idea numérica**: si las columnas de $X$ están
correlacionadas, $X^\top X$ está mal condicionada y la inversión amplifica errores de
redondeo. En la práctica se resuelve el sistema lineal (como hicimos con
`np.linalg.solve`) o, mejor aún, se usa la descomposición QR o SVD de $X$ directamente
(`np.linalg.lstsq`, que es lo que scikit-learn usa por debajo).